In [ ]:
import json
import re
import time
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
# Pasta para os dados brutos coletados
RAW = Path("dados_brutos")
RAW.mkdir(exist_ok=True)

# Pasta para a base tratada (integrada e limpa)
TRATADO = Path("dados_tratados")
TRATADO.mkdir(exist_ok=True)

# Cabeçalho de User-Agent:
HEADERS = {"User-Agent": "UFAM-CienciaDeDados-Trabalho1 (uso academico)"}

PAUSA = 2 # Pausa entre requisições em segundos

# Chave do Google Books API
GOOGLE_BOOKS_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_BOOKS_API_KEY = userdata.get("GOOGLE_BOOKS_API_KEY")
except Exception:
    import getpass
    GOOGLE_BOOKS_API_KEY = getpass.getpass(
        "Chave da Google Books API (Enter para pular): "
    ) or None

print("Ambiente pronto.")
print("pandas:", pd.__version__)
print("Chave da Google Books API configurada:", bool(GOOGLE_BOOKS_API_KEY))

Ambiente pronto.
pandas: 2.2.3
Chave da Google Books API configurada: True


In [ ]:
# Manter um registro que, para cada fonte, informe: de onde (URL exata), quando (data/hora da coleta), e como (método e parâmetros)
registro_proveniencia = []

def registrar_proveniencia(fonte, url, metodo, parametros=None, observacao=None):
    entrada = {
        "fonte": fonte,
        "url": url,
        "data_hora_coleta_utc": datetime.now(timezone.utc).isoformat(),
        "metodo": metodo,
        "parametros": parametros or {},
        "observacao": observacao or "",
    }
    registro_proveniencia.append(entrada)
    return entrada

def salvar_proveniencia(caminho=RAW / "proveniencia.json"):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(registro_proveniencia, f, ensure_ascii=False, indent=2)
    print(f"Proveniência salva em: {caminho} ({len(registro_proveniencia)} entradas)")


In [ ]:
def raspar(url, user_agent="*"):
    # Verifica no robots.txt do site se a URL pode ser raspada.
    base = re.match(r"^(https?://[^/]+)", url).group(1)
    robots_url = urljoin(base, "/robots.txt")

    # Busca o robots.txt manualmente, com o mesmo User-Agent usado nas demais
    # requisições — a Wikimedia devolve 403 para o UA padrão do urllib, e
    # isso faz o RobotFileParser assumir "disallow all" por segurança.
    resp = requests.get(robots_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    rp = RobotFileParser()
    rp.set_url(robots_url)
    rp.parse(resp.text.splitlines())  # em vez de rp.read()

    permitido = rp.can_fetch(user_agent, url)
    print(f"'robots.txt': {robots_url}")
    print(f"Permite acessar {url}? -> {permitido}")
    return permitido, robots_url

URL = "https://pt.wikipedia.org/wiki/Categoria:Livros_adaptados_para_o_cinema"
permitido, robots_url = raspar(URL)

registrar_proveniencia(
    fonte="Wikipedia (Pt) - Verificação do 'robots.txt'",
    url=robots_url,
    metodo="urllib.robotparser",
    observacao=f"can_fetch para {URL} = {permitido}",
)

assert permitido, "Robots.txt não permite o acesso."

'robots.txt': https://pt.wikipedia.org/robots.txt
Permite acessar https://pt.wikipedia.org/wiki/Categoria:Livros_adaptados_para_o_cinema? -> True


**Licença/termos de uso (Wikipédia):** o conteúdo textual da Wikipédia em português é distribuído
sob a licença **CC BY-SA 4.0** (Creative Commons Atribuição-CompartilhaIgual), o que permite uso,
cópia e redistribuição — inclusive para fins acadêmicos e comerciais — desde que se dê a devida
atribuição à fonte e que trabalhos derivados sejam compartilhados sob a mesma licença. Para este
projeto, extraímos apenas os títulos das obras e os links dos artigos (dados factuais, não o texto
integral dos artigos), o que está plenamente dentro dos termos de uso. A atribuição à Wikipédia está
registrada na coluna `url_artigo` da base e no registro de proveniência.

In [ ]:
# WebScraping

DIR_HTML_BRUTO = RAW / "wikipedia_html"
DIR_HTML_BRUTO.mkdir(exist_ok=True)


def baixar_pagina(url, indice_pagina):
    # Baixa uma página da categoria, salva o HTML bruto e retorna o BeautifulSoup.
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    caminho_html = DIR_HTML_BRUTO / f"categoria_pagina_{indice_pagina:02d}.html"
    caminho_html.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="Wikipedia (Pt) - Categoria: Livros_adaptados_para_o_cinema",
        url=resp.url,
        metodo="requests.get + BeautifulSoup",
        parametros={"pagina": indice_pagina, "status_http": resp.status_code},
        observacao=f"HTML bruto salvo em {caminho_html}",
    )

    return BeautifulSoup(resp.text, "html.parser")


def extrair_livros(soup):
    # Extrai (titulo, url_artigo) de cada item listado na página de categoria.
    livros = []
    container = soup.find("div", id="mw-pages")
    if container is None:
        return livros

    for link in container.select("div.mw-category-group ul li a"):
        titulo = link.get("title") or link.text.strip()
        href = link.get("href")
        if not href:
            continue
        url_artigo = urljoin("https://pt.wikipedia.org", href)
        livros.append({"titulo_wikipedia_sujo": titulo, "url_artigo": url_artigo})

    return livros


def proxima_pagina(soup):
    # Procura o link 'página seguinte' da categoria, se existir.
    container = soup.find("div", id="mw-pages")
    if container is None:
        return None
    for a in container.find_all("a"):
        if "página seguinte" in a.text.lower() or "next page" in a.text.lower():
            return urljoin("https://pt.wikipedia.org", a.get("href"))
    return None


In [ ]:
# Loop de coleta com paginação, sem sobrecarregar o servidor.

filmes = []
url_atual = URL
indice = 1

while url_atual:
    print(f"Coletando página {indice}: {url_atual}")
    soup = baixar_pagina(url_atual, indice)
    livros_pagina = extrair_livros(soup)
    print(f"  -> {len(livros_pagina)} títulos extraídos")
    filmes.extend(livros_pagina)

    url_atual = proxima_pagina(soup)
    indice += 1
    if url_atual:
        time.sleep(PAUSA)

print(f"\nTotal de filmes baseado em livros coletados: {len(filmes)}")


Coletando página 1: https://pt.wikipedia.org/wiki/Categoria:Livros_adaptados_para_o_cinema
  -> 200 títulos extraídos
Coletando página 2: https://pt.wikipedia.org/w/index.php?title=Categoria:Livros_adaptados_para_o_cinema&pagefrom=Josefine+Mutzenbacher#mw-pages
  -> 200 títulos extraídos
Coletando página 3: https://pt.wikipedia.org/w/index.php?title=Categoria:Livros_adaptados_para_o_cinema&pagefrom=Um+Antrop%C3%B3logo+em+Marte#mw-pages
  -> 24 títulos extraídos

Total de filmes baseado em livros coletados: 424


In [ ]:
def limpar_titulo(titulo):
    # Remove sufixos de desambiguação tipo '(livro)', '(romance)' etc.
    return re.sub(r"\s*\([^)]*\)\s*$", "", titulo).strip()

df_wikipedia_bruto = pd.DataFrame(filmes).drop_duplicates()
df_wikipedia_bruto["titulo_wikipedia"] = df_wikipedia_bruto["titulo_wikipedia_sujo"].apply(
    limpar_titulo
)

print(df_wikipedia_bruto.shape)
df_wikipedia_bruto.head()


(424, 3)


,titulo_wikipedia_sujo,url_artigo,titulo_wikipedia
0,Artur e os Minimeus (série),https://pt.wikipedia.org/wiki/Artur_e_os_Minim...,Artur e os Minimeus
1,1 Litre no Namida (livro),https://pt.wikipedia.org/wiki/1_Litre_no_Namid...,1 Litre no Namida
2,84 Charing Cross Road,https://pt.wikipedia.org/wiki/84_Charing_Cross...,84 Charing Cross Road
3,2010: Odyssey Two,https://pt.wikipedia.org/wiki/2010:_Odyssey_Two,2010: Odyssey Two
4,3096 Dias,https://pt.wikipedia.org/wiki/3096_Dias,3096 Dias


In [ ]:
# Dado bruto: exatamente como veio da extração, sem qualquer limpeza de conteúdo
caminho_csv_bruto = RAW / "wikipedia_livros_categoria.csv"
df_wikipedia_bruto.to_csv(caminho_csv_bruto, index=False, encoding="utf-8")

salvar_proveniencia()

print(f"Salvo: {caminho_csv_bruto}")


Proveniência salva em: dados_brutos/proveniencia.json (4 entradas)
Salvo: dados_brutos/wikipedia_livros_categoria.csv


**Licença/termos de uso (Google Books API):** o uso da API é regido pelos
[Termos de Serviço das APIs do Google](https://developers.google.com/terms) e pela política de uso
da Books API, que permite consultas para fins de pesquisa, desenvolvimento e uso acadêmico sem custo,
dentro dos limites de cota (requisições por dia). É proibido usar os dados para criar um serviço
concorrente ao Google Books ou redistribuir os dados brutos como produto próprio. Para este trabalho
acadêmico — que usa apenas metadados agregados (nota média, autor, editora, ano) e não redistribui a
API como serviço —, o uso está dentro do permitido.

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# API

GOOGLEBOOKS_BRUTO = RAW / Path("google_books_json")
GOOGLEBOOKS_BRUTO.mkdir(exist_ok=True)

GOOGLE_BOOKS_ENDPOINT = "https://www.googleapis.com/books/v1/volumes"

def consultar_livros(titulo, indice, max_tentativas=3):
    # Consulta a Google Books API por um título e salva o JSON bruto da resposta.
    params = {"q": f"intitle:{titulo}"}
    if GOOGLE_BOOKS_API_KEY:
        params["key"] = GOOGLE_BOOKS_API_KEY

    for tentativa in range(1, max_tentativas + 1):
        resp = requests.get(
            GOOGLE_BOOKS_ENDPOINT, params=params, headers=HEADERS, timeout=30
        )

        if resp.status_code == 429 or resp.status_code == 503:
            # Cota excedida ou Serviço indisponível. Aguarda e tenta novamente.
            print(f"  {resp.status_code} (cota excedida ou serviço indisponível) para '{titulo}', tentativa {tentativa}. Aguardando...")
            time.sleep(PAUSA * 5) # Longer pause for server errors
            continue

        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"Falha ao consultar '{titulo}' após {max_tentativas} tentativas.")

    caminho_json = GOOGLEBOOKS_BRUTO / f"googlebooks_{indice:04d}.json"
    caminho_json.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="Google Books API",
        url=resp.url,
        metodo="requests.get (Google Books API v1/volumes)",
        parametros={"q": params["q"], "status_http": resp.status_code},
        observacao=f"JSON bruto salvo em {caminho_json}",
    )

    return resp.json()


def extrair_info(payload, titulo_wikipedia):
    # Extrai os campos de interesse do primeiro item retornado.
    itens = payload.get("items")
    if not itens:
        return {
            "titulo_wikipedia": titulo_wikipedia,
            "titulo_google_books": None,
            "autor": None,
            "ano_publicacao": None,
            "nota_media": None,
            "n_avaliacoes": None,
            "categorias": None,
            "idioma": None,
            "editora": None,
        }

    info = itens[0].get("volumeInfo", {})
    return {
        "titulo_wikipedia": titulo_wikipedia,
        "titulo_google_books": info.get("title"),
        "autor": ", ".join(info.get("authors", [])) or None,
        "ano_publicacao": info.get("publishedDate"),
        "nota_media": info.get("averageRating"),
        "n_avaliacoes": info.get("ratingsCount"),
        "categorias": ", ".join(info.get("categories", [])) or None,
        "idioma": info.get("language"),
        "editora": info.get("publisher"),
    }

In [ ]:
# Loop de coleta, reaproveitando os títulos já extraídos da Wikipédia

titulos_unicos = df_wikipedia_bruto["titulo_wikipedia"].dropna().unique()

resultados_livros = []

for i, titulo in enumerate(titulos_unicos, start=1):
    print(f"[{i}/{len(titulos_unicos)}] Consultando: {titulo}")
    payload = consultar_livros(titulo, i)
    resultados_livros.append(extrair_info(payload, titulo))
    time.sleep(PAUSA)

[1/424] Consultando: Artur e os Minimeus
[2/424] Consultando: 1 Litre no Namida
[3/424] Consultando: 84 Charing Cross Road
[4/424] Consultando: 2010: Odyssey Two
[5/424] Consultando: 3096 Dias
[6/424] Consultando: A Autobiografia de um Mentiroso: Volume VI
[7/424] Consultando: A Culpa É das Estrelas
[8/424] Consultando: À Espera de um Milagre
[9/424] Consultando: A Feiticeira
[10/424] Consultando: A Filosofia na Alcova
[11/424] Consultando: A Guerra dos Mundos
[12/424] Consultando: A Incendiária
[13/424] Consultando: A Little Princess
[14/424] Consultando: A Most Wanted Man
[15/424] Consultando: A Sangue Frio
[16/424] Consultando: A Vênus das Peles
[17/424] Consultando: About a Boy
[18/424] Consultando: Achados e Perdidos
[19/424] Consultando: The Adventures of Tom Sawyer
[20/424] Consultando: O Advogado do Diabo
[21/424] Consultando: Airport
[22/424] Consultando: Amanhecer
[23/424] Consultando: Amar, Verbo Intransitivo
[24/424] Consultando: Psicopata Americano
[25/424] Consultando: De

In [ ]:
df_google_books_bruto = pd.DataFrame(resultados_livros)
df_google_books_bruto.head(50)

,titulo_wikipedia,titulo_google_books,autor,ano_publicacao,nota_media,n_avaliacoes,categorias,idioma,editora
0,Artur e os Minimeus,Arthur and the Minimoys,Luc Besson,2006-04-25,4.0,1.0,Juvenile Fiction,en,Harper Collins
1,1 Litre no Namida,1 Litre of Tears,Kito Aya,2024-09-11,NaN,NaN,Biography & Autobiography,vi,None
2,84 Charing Cross Road,"84, Charing Cross Road",Helene Hanff,2027-02-02,NaN,NaN,Literary Collections,en,Penguin Classics
3,2010: Odyssey Two,2010: Odyssey Two,Arthur C. Clarke,1984-01-12,4.0,18.0,Fiction,en,Del Rey
4,3096 Dias,"3,096 Days","Natascha Kampusch, Heike Gronemeier, Corinna M...",2010-09-16,NaN,NaN,Biography & Autobiography,en,Penguin UK
5,A Autobiografia de um Mentiroso: Volume VI,Vi-a Mais Sacra,Melkyzedek Siqueira César,2024-05-05,NaN,NaN,Biography & Autobiography,pt-BR,Clube de Autores
6,A Culpa É das Estrelas,A Culpa Não É Das Estrelas,Daniel Carlos,2022-06-04,NaN,NaN,Education,pt-BR,Clube de Autores
7,À Espera de um Milagre,À Espera Da Extraordinária Cura,Marinalva Irenice Conceição; Guilherme Do Espí...,2025-05-05,NaN,NaN,Humor,pt-BR,Clube de Autores
8,A Feiticeira,"Winnie, a Feiticeira","Korky Paul, Valerie Thomas",1998,NaN,NaN,Cats,pt-BR,Martins Martins Fontes
9,A Filosofia na Alcova,"Filosofia na Alcôva, A",marquis de Sade,2000,5.0,1.0,Fiction,pt-BR,Editora Iluminuras Ltda


In [ ]:
# Dado bruto: exatamente como veio da API, sem qualquer limpeza de conteúdo
caminho_csv_gb = RAW / "google_books_resultados.csv"
df_google_books_bruto.to_csv(caminho_csv_gb, index=False, encoding="utf-8")

salvar_proveniencia()  # reescreve o log completo (Wikipedia + Google Books)

print(f"Salvo: {caminho_csv_gb}")

Proveniência salva em: dados_brutos/proveniencia.json (428 entradas)
Salvo: dados_brutos/google_books_resultados.csv


In [ ]:
# Contando o número de valores nulos (NaN) em cada coluna
missing_values = df_google_books_bruto.isnull().sum()
print("Contagem de valores ausentes por coluna:")
print(missing_values[missing_values > 0])

Contagem de valores ausentes por coluna:
titulo_google_books      1
autor                   20
ano_publicacao          11
nota_media             341
n_avaliacoes           341
categorias             125
idioma                   1
editora                182
dtype: int64


In [ ]:
# Calculando a porcentagem de valores ausentes por coluna
missing_percentage = (df_google_books_bruto.isnull().sum() / len(df_google_books_bruto)) * 100
print("\nPorcentagem de valores ausentes por coluna:")
print(missing_percentage[missing_percentage > 0].sort_values(ascending=False))


Porcentagem de valores ausentes por coluna:
nota_media             80.424528
n_avaliacoes           80.424528
editora                42.924528
categorias             29.481132
autor                   4.716981
ano_publicacao          2.594340
titulo_google_books     0.235849
idioma                  0.235849
dtype: float64


## Terceira fonte: TMDB API (dados dos filmes)

Até aqui temos:
- **Wikipédia (scraping):** lista de livros adaptados para o cinema (`titulo_wikipedia`).
- **Google Books API:** avaliação dos leitores sobre o **livro** (`nota_media`, `n_avaliacoes`, etc.).

Falta o lado do **filme**: bilheteria, orçamento, nota do público, popularidade, data de
lançamento. Isso vem da **TMDB (The Movie Database) API** — uma API pública, gratuita para
uso não comercial mediante cadastro, amplamente usada em projetos acadêmicos.

**Como obter a chave (token):**
1. Crie uma conta em https://www.themoviedb.org/ (gratuita).
2. Vá em *Configurações → API* e solicite uma chave (aprovação é automática para uso pessoal/educacional).
3. Você recebe duas credenciais: a **API Key (v3)** e o **Access Token (v4, Bearer)**. Usaremos o
   token v4 no cabeçalho `Authorization`, no mesmo padrão de autenticação "token no header" visto para
   a API do GitHub (Módulo 3 da disciplina).

**Licença/termos de uso:** a TMDB exige que o uso da API cite a atribuição *"This product uses the
TMDB API but is not endorsed or certified by TMDB"* em qualquer produto que a utilize, e proíbe uso
que viole seus Termos de Serviço (ex.: revender os dados brutos como um produto concorrente). Para um
trabalho acadêmico, o uso é permitido.


In [ ]:
# Configuração da TMDB API
TMDB_TOKEN = None
try:
    from google.colab import userdata
    TMDB_TOKEN = userdata.get("TMDB_TOKEN")
except Exception:
    import getpass
    TMDB_TOKEN = getpass.getpass(
        "Token (Bearer v4) da TMDB API (Enter para pular): "
    ) or None

TMDB_ENDPOINT_SEARCH = "https://api.themoviedb.org/3/search/movie"
TMDB_ENDPOINT_DETAILS = "https://api.themoviedb.org/3/movie/{id}"

TMDB_HEADERS = dict(HEADERS)  # reaproveita o User-Agent definido antes
if TMDB_TOKEN:
    TMDB_HEADERS["Authorization"] = f"Bearer {TMDB_TOKEN}"

TMDB_BRUTO = RAW / "tmdb_json"
TMDB_BRUTO.mkdir(exist_ok=True)

print("Chave da TMDB API configurada:", bool(TMDB_TOKEN))


Chave da TMDB API configurada: True


In [ ]:
def buscar_filme_tmdb(titulo, indice, max_tentativas=3):
    # Busca um filme pelo título na TMDB e salva o JSON bruto da resposta de busca.
    params = {"query": titulo, "language": "pt-BR", "include_adult": "false"}

    for tentativa in range(1, max_tentativas + 1):
        resp = requests.get(
            TMDB_ENDPOINT_SEARCH, params=params, headers=TMDB_HEADERS, timeout=30
        )

        if resp.status_code == 429:
            # Cota/limite de requisições excedido
            espera = int(resp.headers.get("Retry-After", PAUSA * 5))
            print(f"  429 (limite excedido) para '{titulo}', tentativa {tentativa}. Aguardando {espera}s...")
            time.sleep(espera)
            continue

        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"Falha ao buscar '{titulo}' na TMDB após {max_tentativas} tentativas.")

    caminho_json = TMDB_BRUTO / f"tmdb_busca_{indice:04d}.json"
    caminho_json.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="TMDB API - busca por filme",
        url=resp.url,
        metodo="requests.get (TMDB /search/movie)",
        parametros={"query": titulo, "status_http": resp.status_code},
        observacao=f"JSON bruto salvo em {caminho_json}",
    )

    return resp.json()


def detalhar_filme_tmdb(tmdb_id, indice, max_tentativas=3):
    # Consulta os detalhes de um filme (orçamento, receita, duração, gêneros)
    # e salva o JSON bruto da resposta.
    url = TMDB_ENDPOINT_DETAILS.format(id=tmdb_id)
    params = {"language": "pt-BR"}

    for tentativa in range(1, max_tentativas + 1):
        resp = requests.get(url, params=params, headers=TMDB_HEADERS, timeout=30)

        if resp.status_code == 429:
            espera = int(resp.headers.get("Retry-After", PAUSA * 5))
            print(f"  429 (limite excedido) para id={tmdb_id}, tentativa {tentativa}. Aguardando {espera}s...")
            time.sleep(espera)
            continue

        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"Falha ao detalhar filme id={tmdb_id} após {max_tentativas} tentativas.")

    caminho_json = TMDB_BRUTO / f"tmdb_detalhes_{indice:04d}.json"
    caminho_json.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="TMDB API - detalhes do filme",
        url=resp.url,
        metodo="requests.get (TMDB /movie/{id})",
        parametros={"tmdb_id": tmdb_id, "status_http": resp.status_code},
        observacao=f"JSON bruto salvo em {caminho_json}",
    )

    return resp.json()


def extrair_info_tmdb(titulo_wikipedia, busca_payload, detalhes_payload):
    # Extrai os campos de interesse combinando a busca (relevância/ordenação)
    # com os detalhes (orçamento, receita, duração, gêneros).
    if detalhes_payload is None:
        return {
            "titulo_wikipedia": titulo_wikipedia,
            "tmdb_id": None,
            "titulo_tmdb": None,
            "data_lancamento": None,
            "idioma_original": None,
            "generos": None,
            "duracao_min": None,
            "orcamento": None,
            "receita": None,
            "popularidade": None,
            "media_votos": None,
            "contagem_votos": None,
        }

    generos = ", ".join(g["name"] for g in detalhes_payload.get("genres", [])) or None

    return {
        "titulo_wikipedia": titulo_wikipedia,
        "tmdb_id": detalhes_payload.get("id"),
        "titulo_tmdb": detalhes_payload.get("title"),
        "data_lancamento": detalhes_payload.get("release_date") or None,
        "idioma_original": detalhes_payload.get("original_language"),
        "generos": generos,
        "duracao_min": detalhes_payload.get("runtime"),
        "orcamento": detalhes_payload.get("budget"),
        "receita": detalhes_payload.get("revenue"),
        "popularidade": detalhes_payload.get("popularity"),
        "media_votos": detalhes_payload.get("vote_average"),
        "contagem_votos": detalhes_payload.get("vote_count"),
    }


In [ ]:
# Loop de coleta, reaproveitando os mesmos títulos já usados no Google Books
resultados_tmdb = []

for i, titulo in enumerate(titulos_unicos, start=1):
    print(f"[{i}/{len(titulos_unicos)}] Buscando na TMDB: {titulo}")

    busca = buscar_filme_tmdb(titulo, i)
    candidatos = busca.get("results") or []

    if not candidatos:
        print("  -> nenhum resultado encontrado")
        resultados_tmdb.append(extrair_info_tmdb(titulo, busca, None))
        time.sleep(PAUSA)
        continue

    # A TMDB já ordena os resultados por relevância/popularidade; usamos o primeiro.
    melhor_candidato = candidatos[0]
    time.sleep(PAUSA)

    detalhes = detalhar_filme_tmdb(melhor_candidato["id"], i)
    resultados_tmdb.append(extrair_info_tmdb(titulo, busca, detalhes))
    time.sleep(PAUSA)

print(f"\nTotal de consultas TMDB realizadas: {len(resultados_tmdb)}")

[1/424] Buscando na TMDB: Artur e os Minimeus
[2/424] Buscando na TMDB: 1 Litre no Namida
[3/424] Buscando na TMDB: 84 Charing Cross Road
[4/424] Buscando na TMDB: 2010: Odyssey Two
[5/424] Buscando na TMDB: 3096 Dias
[6/424] Buscando na TMDB: A Autobiografia de um Mentiroso: Volume VI
  -> nenhum resultado encontrado
[7/424] Buscando na TMDB: A Culpa É das Estrelas
[8/424] Buscando na TMDB: À Espera de um Milagre
[9/424] Buscando na TMDB: A Feiticeira
[10/424] Buscando na TMDB: A Filosofia na Alcova
[11/424] Buscando na TMDB: A Guerra dos Mundos
[12/424] Buscando na TMDB: A Incendiária
[13/424] Buscando na TMDB: A Little Princess
[14/424] Buscando na TMDB: A Most Wanted Man
[15/424] Buscando na TMDB: A Sangue Frio
[16/424] Buscando na TMDB: A Vênus das Peles
  -> nenhum resultado encontrado
[17/424] Buscando na TMDB: About a Boy
[18/424] Buscando na TMDB: Achados e Perdidos
[19/424] Buscando na TMDB: The Adventures of Tom Sawyer
[20/424] Buscando na TMDB: O Advogado do Diabo
[21/424] 

In [ ]:
df_tmdb_bruto = pd.DataFrame(resultados_tmdb)

print(df_tmdb_bruto.shape)
df_tmdb_bruto.head(10)

(424, 12)


,titulo_wikipedia,tmdb_id,titulo_tmdb,data_lancamento,idioma_original,generos,duracao_min,orcamento,receita,popularidade,media_votos,contagem_votos
0,Artur e os Minimeus,9992.0,Arthur e os Minimoys,2006-12-13,fr,"Aventura, Fantasia, Animação, Família",94.0,86000000.0,108605609.0,6.5825,6.394,3100.0
1,1 Litre no Namida,41261.0,1リットルの涙,2005-02-05,ja,Drama,98.0,0.0,0.0,1.9354,7.833,36.0
2,84 Charing Cross Road,15677.0,"Nunca Te Vi, Sempre Te Amei",1987-02-13,en,"Drama, Romance",100.0,0.0,2538291.0,2.9330,7.200,232.0
3,2010: Odyssey Two,4437.0,2010: O Ano em que Faremos Contato,1984-12-06,en,"Thriller, Ficção científica",115.0,28000000.0,40400000.0,12.2490,6.662,1117.0
4,3096 Dias,166666.0,3096 Dias de Cativeiro,2013-02-21,de,Drama,111.0,0.0,6677474.0,8.3553,7.398,1022.0
5,A Autobiografia de um Mentiroso: Volume VI,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
6,A Culpa É das Estrelas,222935.0,A Culpa é das Estrelas,2014-06-03,en,"Romance, Drama",132.0,14000000.0,307166834.0,8.7213,7.590,11686.0
7,À Espera de um Milagre,497.0,À Espera de um Milagre,1999-12-10,en,"Fantasia, Drama, Crime",189.0,60000000.0,286801374.0,35.0176,8.506,19795.0
8,A Feiticeira,9722.0,A Feiticeira,2005-06-21,en,"Comédia, Fantasia, Romance",102.0,85000000.0,131426169.0,5.7212,5.091,1689.0
9,A Filosofia na Alcova,486576.0,A Filosofia na Alcova,2017-11-23,pt,Drama,77.0,0.0,0.0,0.9440,5.625,8.0


In [ ]:
# Dado bruto: exatamente como veio da API, sem qualquer limpeza de conteúdo
caminho_csv_tmdb = RAW / "tmdb_resultados.csv"
df_tmdb_bruto.to_csv(caminho_csv_tmdb, index=False, encoding="utf-8")

salvar_proveniencia()  # reescreve o log completo (Wikipedia + Google Books + TMDB)

print(f"Salvo: {caminho_csv_tmdb}")


Proveniência salva em: dados_brutos/proveniencia.json (1216 entradas)
Salvo: dados_brutos/tmdb_resultados.csv


In [ ]:
# Profiling inicial da TMDB (mesmo padrão aplicado ao Google Books)
print("Dimensões:", df_tmdb_bruto.shape)
print("\nTipos de dados:")
print(df_tmdb_bruto.dtypes)

missing_values_tmdb = df_tmdb_bruto.isnull().sum()
print("\nContagem de valores ausentes por coluna:")
print(missing_values_tmdb[missing_values_tmdb > 0])

missing_percentage_tmdb = (df_tmdb_bruto.isnull().sum() / len(df_tmdb_bruto)) * 100
print("\nPorcentagem de valores ausentes por coluna:")
print(missing_percentage_tmdb[missing_percentage_tmdb > 0].sort_values(ascending=False))

taxa_match = df_tmdb_bruto["tmdb_id"].notna().mean() * 100
print(f"\nTaxa de correspondência (título encontrou um filme na TMDB): {taxa_match:.1f}%")


Dimensões: (424, 12)

Tipos de dados:
titulo_wikipedia     object
tmdb_id             float64
titulo_tmdb          object
data_lancamento      object
idioma_original      object
generos              object
duracao_min         float64
orcamento           float64
receita             float64
popularidade        float64
media_votos         float64
contagem_votos      float64
dtype: object

Contagem de valores ausentes por coluna:
tmdb_id            60
titulo_tmdb        60
data_lancamento    66
idioma_original    60
generos            66
duracao_min        60
orcamento          60
receita            60
popularidade       60
media_votos        60
contagem_votos     60
dtype: int64

Porcentagem de valores ausentes por coluna:
data_lancamento    15.566038
generos            15.566038
tmdb_id            14.150943
titulo_tmdb        14.150943
idioma_original    14.150943
duracao_min        14.150943
orcamento          14.150943
receita            14.150943
popularidade       14.150943
media_vot

### Candidatos por consulta às APIs

Antes de partir para a limpeza, registramos quantos resultados cada consulta ao Google Books e
à TMDB retornou. Como o código mantém apenas o primeiro, esta contagem torna visível quantas
edições/candidatos foram descartados, informação para o dataset card.

In [ ]:
# Quantos candidatos cada consulta às APIs devolveu.
# extrair_info() e a busca da TMDB ficam com o primeiro resultado e descartam o resto.
# Recontamos, a partir dos JSONs brutos salvos, quantos candidatos havia por título, tornando visível esse descarte para registrar no dataset card.
def _contar_resultados(pasta, chave_lista, padrao="*.json"):
    contagens = []
    for arq in sorted(Path(pasta).glob(padrao)):
        try:
            payload = json.loads(arq.read_text(encoding="utf-8"))
        except Exception:
            continue
        n = len(payload.get(chave_lista, []) or [])
        contagens.append(n)
    return pd.Series(contagens, name="n_resultados")

for nome, pasta, chave, padrao in [
    ("Google Books", GOOGLEBOOKS_BRUTO, "items",   "*.json"),
    ("TMDB (busca)", TMDB_BRUTO,        "results", "tmdb_busca_*.json"),
]:
    s = _contar_resultados(pasta, chave, padrao)
    if len(s):
        print(f"--- {nome} ---")
        print(f"  consultas com 0 resultados:  {(s == 0).sum()}")
        print(f"  consultas com 1 resultado:   {(s == 1).sum()}")
        print(f"  consultas com >1 resultado:  {(s > 1).sum()}  "
              f"(estas tiveram candidatos descartados por [0])")
        print(f"  máximo de candidatos p/ 1 título: {s.max()}\n")
    else:
        print(f"--- {nome} ---  (nenhum JSON encontrado em {pasta})\n")

--- Google Books ---
  consultas com 0 resultados:  1
  consultas com 1 resultado:   4
  consultas com >1 resultado:  419  (estas tiveram candidatos descartados por [0])
  máximo de candidatos p/ 1 título: 10

--- TMDB (busca) ---
  consultas com 0 resultados:  60
  consultas com 1 resultado:   130
  consultas com >1 resultado:  234  (estas tiveram candidatos descartados por [0])
  máximo de candidatos p/ 1 título: 20



## Limpeza, tratamento e integração das três fontes

Agora que temos os três brutos coletados e preservados (`dados_brutos/`), seguimos o mesmo fluxo
de limpeza usado em aula: **profiling → dados ausentes → duplicatas → tipos → texto → integração →
outliers → validação → salvar**. Nenhuma célula abaixo altera os DataFrames `*_bruto`: sempre
trabalhamos sobre cópias (`.copy()`), preservando a fonte original intocada em memória e em disco.


### 1. Profiling inicial das três fontes brutas

Antes de qualquer decisão de limpeza, olhamos a "cena do crime" nas três tabelas.


In [ ]:
for nome, dframe in [
    ("Wikipédia (livros->filmes)", df_wikipedia_bruto),
    ("Google Books", df_google_books_bruto),
    ("TMDB", df_tmdb_bruto),
]:
    print("=" * 60)
    print(nome)
    print("=" * 60)
    print(f"Dimensões: {dframe.shape}")
    dframe.info()
    print()


Wikipédia (livros->filmes)
Dimensões: (424, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   titulo_wikipedia_sujo  424 non-null    object
 1   url_artigo             424 non-null    object
 2   titulo_wikipedia       424 non-null    object
dtypes: object(3)
memory usage: 10.1+ KB

Google Books
Dimensões: (424, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   titulo_wikipedia     424 non-null    object 
 1   titulo_google_books  423 non-null    object 
 2   autor                404 non-null    object 
 3   ano_publicacao       413 non-null    object 
 4   nota_media           83 non-null     float64
 5   n_avaliacoes         83 non-null     float64
 6   categorias

### 2. Chave de integração: normalização dos títulos

As três fontes usam `titulo_wikipedia` como referência, mas o texto pode variar levemente entre
fontes (acentuação, caixa, espaços, pontuação residual do scraping). Para o `join` funcionar de
verdade, criamos uma **chave normalizada** (`chave_titulo`): minúsculas, sem acentos, sem pontuação
e sem espaços duplicados. Essa é a mesma ideia de "strip + title/lower" vista em aula para
categóricas sujas, aplicada aqui à chave de integração.


In [ ]:
import unicodedata

def normalizar_titulo(titulo):
    # Gera uma chave de integração robusta a pequenas divergências de grafia:
    # remove acentos, pontuação e normaliza espaços/caixa.
    if pd.isna(titulo):
        return None
    texto = str(titulo).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))  # remove acentos
    texto = re.sub(r"[^a-z0-9\s]", "", texto)   # remove pontuação/símbolos
    texto = re.sub(r"\s+", " ", texto).strip()  # normaliza espaços
    return texto

# Aplicando a chave nas três fontes (sobre cópias, dados brutos preservados)
df_wikipedia = df_wikipedia_bruto.copy()
df_google_books = df_google_books_bruto.copy()
df_tmdb = df_tmdb_bruto.copy()

for dframe in (df_wikipedia, df_google_books, df_tmdb):
    dframe["chave_titulo"] = dframe["titulo_wikipedia"].apply(normalizar_titulo)

# Checagem: a normalização de fato reduziu variações? (esperado: poucas ou nenhuma mudança
# no wikipedia, já que a chave nasce dele; mas serve de checagem sanitária)
print("Títulos únicos (wikipedia) antes de normalizar:", df_wikipedia_bruto["titulo_wikipedia"].nunique())
print("Chaves únicas (wikipedia) após normalizar:      ", df_wikipedia["chave_titulo"].nunique())


Títulos únicos (wikipedia) antes de normalizar: 424
Chaves únicas (wikipedia) após normalizar:       424


### 2.1 Colisões na chave de integração

Verificamos quais títulos, se houver, colapsam na mesma `chave_titulo` após a normalização,
distinguindo reedições (mesma obra) de homônimos (obras distintas).

In [ ]:

colisoes = (
    df_wikipedia["chave_titulo"]
    .value_counts()
    .loc[lambda s: s > 1]
)
print(f"Chaves com colisão na Wikipédia: {len(colisoes)}")
if len(colisoes):
    print("\nTítulos que colapsam na mesma chave (verificar se são a mesma obra):")
    for chave in colisoes.index:
        titulos = df_wikipedia.loc[
            df_wikipedia["chave_titulo"] == chave, "titulo_wikipedia"
        ].tolist()
        print(f"  [{chave}] -> {titulos}")
else:
    print("Nenhuma colisão: a unicidade vem do desenho da coleta, não desta etapa "
          "(registrar isso honestamente no dataset card).")

Chaves com colisão na Wikipédia: 0
Nenhuma colisão: a unicidade vem do desenho da coleta, não desta etapa (registrar isso honestamente no dataset card).


### 3. Duplicatas

Duplicatas completas (linha idêntica em todas as colunas) e duplicatas na **chave de integração**
são tratadas separadamente: a primeira indica erro de coleta; a segunda pode indicar reedições do
mesmo livro (ex.: capa dura e brochura) que, para este projeto, representam a mesma obra — mantemos
apenas a primeira ocorrência por chave.


In [ ]:
for nome, dframe in [("Wikipédia", df_wikipedia), ("Google Books", df_google_books), ("TMDB", df_tmdb)]:
    dups_completas = dframe.duplicated().sum()
    dups_chave = dframe.duplicated(subset=["chave_titulo"]).sum()
    print(f"{nome:15s} | duplicatas completas: {dups_completas:3d} | duplicatas na chave: {dups_chave:3d}")

# Removendo duplicatas na chave, mantendo a primeira ocorrência
df_wikipedia = df_wikipedia.drop_duplicates(subset=["chave_titulo"], keep="first")
df_google_books = df_google_books.drop_duplicates(subset=["chave_titulo"], keep="first")
df_tmdb = df_tmdb.drop_duplicates(subset=["chave_titulo"], keep="first")

print("\nLinhas após remoção de duplicatas na chave:")
print(f"  Wikipédia:    {len(df_wikipedia)}")
print(f"  Google Books: {len(df_google_books)}")
print(f"  TMDB:         {len(df_tmdb)}")


Wikipédia       | duplicatas completas:   0 | duplicatas na chave:   0
Google Books    | duplicatas completas:   0 | duplicatas na chave:   0
TMDB            | duplicatas completas:   0 | duplicatas na chave:   0

Linhas após remoção de duplicatas na chave:
  Wikipédia:    424
  Google Books: 424
  TMDB:         424


### 4. Correção de tipos de dados

`ano_publicacao` (Google Books) vem em formatos mistos (`"1999"`, `"1999-05-01"`, `"1999-05"`).
`data_lancamento` (TMDB) vem como `"AAAA-MM-DD"`. Extraímos o **ano** de cada um para permitir
comparações numéricas, e convertemos os campos numéricos que a API devolveu como texto/objeto.


In [ ]:
# --- Google Books: extraindo o ano de publicação do livro ---
df_google_books["ano_publicacao_livro"] = pd.to_numeric(
    df_google_books["ano_publicacao"].astype(str).str.extract(r"(\d{4})")[0],
    errors="coerce",
)

df_google_books["nota_media"] = pd.to_numeric(df_google_books["nota_media"], errors="coerce")
df_google_books["n_avaliacoes"] = pd.to_numeric(df_google_books["n_avaliacoes"], errors="coerce")

# --- TMDB: convertendo data de lançamento e extraindo o ano do filme ---
df_tmdb["data_lancamento"] = pd.to_datetime(df_tmdb["data_lancamento"], errors="coerce")
df_tmdb["ano_lancamento_filme"] = df_tmdb["data_lancamento"].dt.year

for coluna in ["duracao_min", "orcamento", "receita", "popularidade", "media_votos", "contagem_votos"]:
    df_tmdb[coluna] = pd.to_numeric(df_tmdb[coluna], errors="coerce")

print("Tipos após correção (Google Books):")
print(df_google_books[["ano_publicacao_livro", "nota_media", "n_avaliacoes"]].dtypes)
print("\nTipos após correção (TMDB):")
print(df_tmdb[["data_lancamento", "ano_lancamento_filme", "orcamento", "receita"]].dtypes)


Tipos após correção (Google Books):
ano_publicacao_livro    float64
nota_media              float64
n_avaliacoes            float64
dtype: object

Tipos após correção (TMDB):
data_lancamento         datetime64[ns]
ano_lancamento_filme           float64
orcamento                      float64
receita                        float64
dtype: object


### 5. Dados ausentes

Cada coluna recebe uma decisão justificada, e não um `fillna(0)` genérico:

- **`tmdb_id` ausente** (título não encontrado na TMDB): mantemos a linha, mas criamos a flag
  `tem_match_tmdb` — remover essas linhas jogaria fora casos legítimos de "livro sem adaptação
  encontrada pela API" (por título divergente, filme muito antigo/obscuro etc.), o que é, em si,
  uma informação relevante para a pergunta motivadora.
- **`nota_media`/`n_avaliacoes` (Google Books) ausentes**: mesma lógica — o livro existe, mas não
  tem avaliação suficiente no Google Books. Mantemos `NaN` (não inventamos uma nota) e criamos a
  flag `tem_avaliacoes_google_books`.
- **`orcamento`/`receita` (TMDB) iguais a 0**: na TMDB, filmes sem esse dado divulgado aparecem
  como `0`, não como ausente — tratamos `0` como "não informado" (`NaN`), para não distorcer médias
  e detecção de outliers.
- **Categóricas textuais ausentes** (`generos`, `idioma_original`, `categorias`, `editora`):
  preenchidas com o rótulo explícito `"Não informado"`, preservando a linha sem fabricar uma
  categoria específica.


In [ ]:
# Flags de correspondência/disponibilidade (antes de qualquer imputação)
df_tmdb["tem_match_tmdb"] = df_tmdb["tmdb_id"].notna()
df_google_books["tem_avaliacoes_google_books"] = df_google_books["n_avaliacoes"].notna()

# Orçamento/receita == 0 na TMDB geralmente significa "não divulgado", não "gratuito"
df_tmdb["orcamento"] = df_tmdb["orcamento"].replace(0, np.nan)
df_tmdb["receita"] = df_tmdb["receita"].replace(0, np.nan)
df_tmdb["duracao_min"] = df_tmdb["duracao_min"].replace(0, np.nan)

# Imputação de categóricas com rótulo explícito
for coluna in ["generos", "idioma_original"]:
    df_tmdb[coluna] = df_tmdb[coluna].fillna("Não informado")

for coluna in ["categorias", "editora", "idioma", "autor"]:
    if coluna in df_google_books.columns:
        df_google_books[coluna] = df_google_books[coluna].fillna("Não informado")

print("Resumo de ausentes após as decisões acima:\n")
for nome, dframe in [("Google Books", df_google_books), ("TMDB", df_tmdb)]:
    nulos = dframe.isnull().sum()
    print(f"--- {nome} ---")
    print(nulos[nulos > 0])
    print()


Resumo de ausentes após as decisões acima:

--- Google Books ---
titulo_google_books       1
ano_publicacao           11
nota_media              341
n_avaliacoes            341
ano_publicacao_livro     13
dtype: int64

--- TMDB ---
tmdb_id                  60
titulo_tmdb              60
data_lancamento          66
duracao_min              69
orcamento               231
receita                 228
popularidade             60
media_votos              60
contagem_votos           60
ano_lancamento_filme     66
dtype: int64



### 6. Outliers (bilheteria, orçamento e número de avaliações)

Dados de bilheteria/orçamento são naturalmente assimétricos: a maioria dos filmes fatura pouco e
uma minoria de blockbusters puxa a média para cima — isso não é "erro", é a distribuição real do
mercado de cinema. Por isso, **detectamos** os outliers com o método IQR (mesma técnica da aula),
mas a decisão aqui é **não aplicar capping** aos valores financeiros: um blockbuster de verdade é
um dado real e relevante para a pergunta motivadora ("livros populares viram filmes populares?").
Documentamos a detecção para justificar essa escolha no dataset card.


In [ ]:
def detectar_outliers_iqr(serie):
    # Retorna a máscara booleana de outliers e os limites, pelo método IQR clássico.
    dados_validos = serie.dropna()
    q1 = dados_validos.quantile(0.25)
    q3 = dados_validos.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    mascara = (serie < limite_inferior) | (serie > limite_superior)
    return mascara.fillna(False), limite_inferior, limite_superior


for nome_coluna, dframe in [
    ("n_avaliacoes", df_google_books),
    ("receita", df_tmdb),
    ("orcamento", df_tmdb),
    ("popularidade", df_tmdb),
]:
    mascara, lim_inf, lim_sup = detectar_outliers_iqr(dframe[nome_coluna])
    print(
        f"{nome_coluna:15s} | limites IQR: [{lim_inf:,.1f}, {lim_sup:,.1f}] "
        f"| outliers: {mascara.sum()} de {dframe[nome_coluna].notna().sum()} valores válidos"
    )




n_avaliacoes    | limites IQR: [-8.0, 16.0] | outliers: 13 de 83 valores válidos
receita         | limites IQR: [-218,151,859.4, 392,319,765.6] | outliers: 23 de 196 valores válidos
orcamento       | limites IQR: [-72,000,000.0, 139,200,000.0] | outliers: 10 de 193 valores válidos
popularidade    | limites IQR: [-9.2, 18.9] | outliers: 31 de 364 valores válidos


### 7. Integração (join) das três fontes

A chave de integração é `chave_titulo` (título normalizado). Usamos `how="left"` a partir da
Wikipédia — ela é o "esqueleto" da base (todo livro adaptado que encontramos na categoria), e as
outras duas fontes o enriquecem. Linhas sem correspondência ficam com `NaN` nas colunas daquela
fonte, e já sinalizadas pelas flags criadas no passo 5.


In [ ]:
colunas_wikipedia = ["chave_titulo", "titulo_wikipedia", "url_artigo"]
colunas_google_books = [
    "chave_titulo", "titulo_google_books", "autor", "ano_publicacao_livro",
    "nota_media", "n_avaliacoes", "categorias", "idioma", "editora",
    "tem_avaliacoes_google_books",
]
colunas_tmdb = [
    "chave_titulo", "titulo_tmdb", "data_lancamento", "ano_lancamento_filme",
    "idioma_original", "generos", "duracao_min", "orcamento", "receita",
    "popularidade", "media_votos", "contagem_votos", "tem_match_tmdb",
]

df_integrado = (
    df_wikipedia[colunas_wikipedia]
    .merge(df_google_books[colunas_google_books], on="chave_titulo", how="left")
    .merge(df_tmdb[colunas_tmdb], on="chave_titulo", how="left")
)

# Linhas que vieram sem nenhuma correspondência na Google Books recebem tem_avaliacoes_google_books=False
df_integrado["tem_avaliacoes_google_books"] = df_integrado["tem_avaliacoes_google_books"].fillna(False)
df_integrado["tem_match_tmdb"] = df_integrado["tem_match_tmdb"].fillna(False)

print("Dimensões da base integrada:", df_integrado.shape)
df_integrado.head(10)


Dimensões da base integrada: (424, 24)


,chave_titulo,titulo_wikipedia,url_artigo,titulo_google_books,autor,ano_publicacao_livro,nota_media,n_avaliacoes,categorias,idioma,...,ano_lancamento_filme,idioma_original,generos,duracao_min,orcamento,receita,popularidade,media_votos,contagem_votos,tem_match_tmdb
0,artur e os minimeus,Artur e os Minimeus,https://pt.wikipedia.org/wiki/Artur_e_os_Minim...,Arthur and the Minimoys,Luc Besson,2006.0,4.0,1.0,Juvenile Fiction,en,...,2006.0,fr,"Aventura, Fantasia, Animação, Família",94.0,86000000.0,108605609.0,6.5825,6.394,3100.0,True
1,1 litre no namida,1 Litre no Namida,https://pt.wikipedia.org/wiki/1_Litre_no_Namid...,1 Litre of Tears,Kito Aya,2024.0,NaN,NaN,Biography & Autobiography,vi,...,2005.0,ja,Drama,98.0,NaN,NaN,1.9354,7.833,36.0,True
2,84 charing cross road,84 Charing Cross Road,https://pt.wikipedia.org/wiki/84_Charing_Cross...,"84, Charing Cross Road",Helene Hanff,2027.0,NaN,NaN,Literary Collections,en,...,1987.0,en,"Drama, Romance",100.0,NaN,2538291.0,2.9330,7.200,232.0,True
3,2010 odyssey two,2010: Odyssey Two,https://pt.wikipedia.org/wiki/2010:_Odyssey_Two,2010: Odyssey Two,Arthur C. Clarke,1984.0,4.0,18.0,Fiction,en,...,1984.0,en,"Thriller, Ficção científica",115.0,28000000.0,40400000.0,12.2490,6.662,1117.0,True
4,3096 dias,3096 Dias,https://pt.wikipedia.org/wiki/3096_Dias,"3,096 Days","Natascha Kampusch, Heike Gronemeier, Corinna M...",2010.0,NaN,NaN,Biography & Autobiography,en,...,2013.0,de,Drama,111.0,NaN,6677474.0,8.3553,7.398,1022.0,True
5,a autobiografia de um mentiroso volume vi,A Autobiografia de um Mentiroso: Volume VI,https://pt.wikipedia.org/wiki/A_Autobiografia_...,Vi-a Mais Sacra,Melkyzedek Siqueira César,2024.0,NaN,NaN,Biography & Autobiography,pt-BR,...,NaN,Não informado,Não informado,NaN,NaN,NaN,NaN,NaN,NaN,False
6,a culpa e das estrelas,A Culpa É das Estrelas,https://pt.wikipedia.org/wiki/A_Culpa_%C3%89_d...,A Culpa Não É Das Estrelas,Daniel Carlos,2022.0,NaN,NaN,Education,pt-BR,...,2014.0,en,"Romance, Drama",132.0,14000000.0,307166834.0,8.7213,7.590,11686.0,True
7,a espera de um milagre,À Espera de um Milagre,https://pt.wikipedia.org/wiki/%C3%80_Espera_de...,À Espera Da Extraordinária Cura,Marinalva Irenice Conceição; Guilherme Do Espí...,2025.0,NaN,NaN,Humor,pt-BR,...,1999.0,en,"Fantasia, Drama, Crime",189.0,60000000.0,286801374.0,35.0176,8.506,19795.0,True
8,a feiticeira,A Feiticeira,https://pt.wikipedia.org/wiki/A_Feiticeira_(li...,"Winnie, a Feiticeira","Korky Paul, Valerie Thomas",1998.0,NaN,NaN,Cats,pt-BR,...,2005.0,en,"Comédia, Fantasia, Romance",102.0,85000000.0,131426169.0,5.7212,5.091,1689.0,True
9,a filosofia na alcova,A Filosofia na Alcova,https://pt.wikipedia.org/wiki/A_Filosofia_na_A...,"Filosofia na Alcôva, A",marquis de Sade,2000.0,5.0,1.0,Fiction,pt-BR,...,2017.0,pt,Drama,77.0,NaN,NaN,0.9440,5.625,8.0,True


### 7.1 Validação da confiabilidade do casamento

O join une as fontes por título, mas isso não garante que livro e filme sejam a mesma obra.
Medimos a similaridade entre os títulos e a coerência de datas para marcar, em `match_confiavel`,
quais linhas podem ser usadas com segurança quando a precisão do casamento for crítica.

In [ ]:
# Validação da confiabilidade do casamento entre as fontes.
# O join casa por título normalizado, mas o primeiro resultado de cada API pode não ser a mesma
# obra (homônimos, traduções, reedições). Sem alterar o join, criamos evidências de confiança:
# similaridade entre os títulos das fontes e coerência de anos, consolidadas na flag match_confiavel.
try:
    from rapidfuzz import fuzz
except ImportError:
    !pip install rapidfuzz -q
    from rapidfuzz import fuzz

def _sim(a, b):
    # Similaridade 0–100 robusta a ordem de palavras; None quando falta título.
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return fuzz.token_sort_ratio(str(a).lower(), str(b).lower())

# Similaridade título-a-título contra a Wikipédia (o esqueleto)
df_integrado["sim_titulo_gb"]   = df_integrado.apply(
    lambda r: _sim(r["titulo_wikipedia"], r["titulo_google_books"]), axis=1)
df_integrado["sim_titulo_tmdb"] = df_integrado.apply(
    lambda r: _sim(r["titulo_wikipedia"], r["titulo_tmdb"]), axis=1)

# Coerência de anos: um livro adaptado costuma ser publicado ANTES do filme.
# Tolerância de +3 anos cobre reedições/novelizações próximas ao lançamento.
df_integrado["ano_coerente"] = (
    df_integrado["ano_publicacao_livro"] <= df_integrado["ano_lancamento_filme"] + 3
)

# Flag final: TMDB casou, título do filme razoavelmente parecido, e ano coerente.
# (o lado Google Books entra como reforço, não como bloqueio, pois tem cobertura baixa)
LIMIAR_SIM = 60
df_integrado["match_confiavel"] = (
    df_integrado["tem_match_tmdb"]
    & (df_integrado["sim_titulo_tmdb"] >= LIMIAR_SIM)
    & (df_integrado["ano_coerente"].fillna(True))  # sem ano do livro, não penaliza
)

print("Taxa de match confiável: "
      f"{df_integrado['match_confiavel'].mean()*100:.1f}%  "
      f"({df_integrado['match_confiavel'].sum()} de {len(df_integrado)})")
print("\nDistribuição da similaridade de título (TMDB):")
print(df_integrado["sim_titulo_tmdb"].describe())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.5 MB/s eta 0:00:00
Taxa de match confiável: 21.7%  (92 de 424)

Distribuição da similaridade de título (TMDB):
count    364.000000
mean      67.739434
std       31.014593
min        0.000000
25%       38.844086
50%       68.432056
75%      100.000000
max      100.000000
Name: sim_titulo_tmdb, dtype: float64


### 8. Engenharia de features

Duas variáveis derivadas, diretamente ligadas às perguntas motivadoras do projeto:

- **`anos_ate_adaptacao`**: quanto tempo (em anos) passou entre a publicação do livro e o
  lançamento do filme. Só é calculável quando ambas as datas existem.
- **`retorno_financeiro`**: proporção `(receita - orçamento) / orçamento`, quando o orçamento é
  conhecido e positivo — uma medida de desempenho comercial do filme.


In [ ]:
df_integrado["retorno_financeiro"] = np.where(
    df_integrado["orcamento"] > 0,
    (df_integrado["receita"] - df_integrado["orcamento"]) / df_integrado["orcamento"],
    np.nan,
)

display(df_integrado[["titulo_wikipedia", "retorno_financeiro"]].describe())


,retorno_financeiro
count,1.770000e+02
mean,1.413008e+04
std,1.879110e+05
min,-9.822581e-01
25%,4.400000e-01
50%,2.219957e+00
75%,6.125000e+00
max,2.499999e+06


### 9. Validação final e comparação bruto × tratado

Fechamos com o "recibo" da limpeza: dimensões, tipos e contagem de nulos antes/depois, por fonte.


In [ ]:
print("=" * 60)
print("RESUMO POR FONTE (bruto -> após limpeza/join)")
print("=" * 60)
print(f"Wikipédia:    {df_wikipedia_bruto.shape[0]:>4} linhas brutas -> {df_wikipedia.shape[0]:>4} após dedup. de chave")
print(f"Google Books: {df_google_books_bruto.shape[0]:>4} linhas brutas -> {df_google_books.shape[0]:>4} após dedup. de chave")
print(f"TMDB:         {df_tmdb_bruto.shape[0]:>4} linhas brutas -> {df_tmdb.shape[0]:>4} após dedup. de chave")

print("\n" + "=" * 60)
print("BASE INTEGRADA FINAL")
print("=" * 60)
print(f"Dimensões:              {df_integrado.shape}")
print(f"Taxa de match TMDB:     {df_integrado['tem_match_tmdb'].mean() * 100:.1f}%")
print(f"Taxa c/ nota Google Books: {df_integrado['tem_avaliacoes_google_books'].mean() * 100:.1f}%")
print("\nValores ausentes por coluna:")
print(df_integrado.isnull().sum().sort_values(ascending=False))


RESUMO POR FONTE (bruto -> após limpeza/join)
Wikipédia:     424 linhas brutas ->  424 após dedup. de chave
Google Books:  424 linhas brutas ->  424 após dedup. de chave
TMDB:          424 linhas brutas ->  424 após dedup. de chave

BASE INTEGRADA FINAL
Dimensões:              (424, 29)
Taxa de match TMDB:     85.8%
Taxa c/ nota Google Books: 19.6%

Valores ausentes por coluna:
nota_media                     341
n_avaliacoes                   341
retorno_financeiro             247
orcamento                      231
receita                        228
duracao_min                     69
ano_lancamento_filme            66
data_lancamento                 66
media_votos                     60
popularidade                    60
titulo_tmdb                     60
contagem_votos                  60
sim_titulo_tmdb                 60
ano_publicacao_livro            13
titulo_google_books              1
sim_titulo_gb                    1
categorias                       0
chave_titulo            

### 9.1 Validação de sanidade da base final

Conferência automática de que os valores estão em faixas plausíveis (notas, anos, durações,
valores financeiros) antes de salvar. Não interrompe a execução: lista todas as checagens.

In [ ]:

def checar(condicao_ok, descricao):
    n_falhas = (~condicao_ok).sum()
    status = "OK " if n_falhas == 0 else f"FALHOU ({n_falhas} linhas)"
    print(f"[{status}] {descricao}")

d = df_integrado
print("Validação de faixas (linhas com valor presente):\n")
checar(d["nota_media"].dropna().between(0, 5).reindex(d.index, fill_value=True),
       "nota_media entre 0 e 5")
checar(d["media_votos"].dropna().between(0, 10).reindex(d.index, fill_value=True),
       "media_votos (TMDB) entre 0 e 10")
checar((d["receita"].fillna(0) >= 0),
       "receita não-negativa")
checar((d["orcamento"].fillna(0) >= 0),
       "orcamento não-negativo")
checar(d["ano_lancamento_filme"].dropna().between(1888, 2027).reindex(d.index, fill_value=True),
       "ano do filme entre 1888 (1º filme) e 2027")
checar(d["ano_publicacao_livro"].dropna().between(1400, 2027).reindex(d.index, fill_value=True),
       "ano do livro entre 1400 e 2027")
checar(d["duracao_min"].dropna().between(1, 600).reindex(d.index, fill_value=True),
       "duração entre 1 e 600 min")

print(f"\nDimensões finais: {d.shape}")
print(f"Colunas: {list(d.columns)}")

Validação de faixas (linhas com valor presente):

[OK ] nota_media entre 0 e 5
[OK ] media_votos (TMDB) entre 0 e 10
[OK ] receita não-negativa
[OK ] orcamento não-negativo
[OK ] ano do filme entre 1888 (1º filme) e 2027
[OK ] ano do livro entre 1400 e 2027
[OK ] duração entre 1 e 600 min

Dimensões finais: (424, 29)
Colunas: ['chave_titulo', 'titulo_wikipedia', 'url_artigo', 'titulo_google_books', 'autor', 'ano_publicacao_livro', 'nota_media', 'n_avaliacoes', 'categorias', 'idioma', 'editora', 'tem_avaliacoes_google_books', 'titulo_tmdb', 'data_lancamento', 'ano_lancamento_filme', 'idioma_original', 'generos', 'duracao_min', 'orcamento', 'receita', 'popularidade', 'media_votos', 'contagem_votos', 'tem_match_tmdb', 'sim_titulo_gb', 'sim_titulo_tmdb', 'ano_coerente', 'match_confiavel', 'retorno_financeiro']


9.2 Dicionário de variáveis (dataset card A.3)
Ficha padronizada da base tratada, uma linha por variável. Também exportada para dados_tratados/dicionario_variaveis.csv, para compor o dataset card.

In [ ]:

dicionario = [
    ("chave_titulo",        "texto",             "Título normalizado (sem acento/caixa/pontuação) usado como chave de integração", "-"),
    ("titulo_wikipedia",    "texto",             "Título da obra como listado na categoria da Wikipédia", "-"),
    ("url_artigo",          "texto",             "URL do artigo na Wikipédia (atribuição da fonte)", "-"),
    ("titulo_google_books", "texto",             "Título do livro conforme a Google Books API", "-"),
    ("autor",               "categórica",        "Autor(es) do livro (Google Books)", "-"),
    ("ano_publicacao_livro","numérica discreta", "Ano de publicação da edição retornada pelo Google Books", "ano"),
    ("nota_media",          "numérica contínua", "Nota média do livro no Google Books", "escala 0–5"),
    ("n_avaliacoes",        "numérica discreta", "Número de avaliações do livro no Google Books", "contagem"),
    ("categorias",          "categórica",        "Categorias/gêneros do livro (Google Books)", "-"),
    ("idioma",              "categórica",        "Idioma da edição do livro (Google Books)", "-"),
    ("editora",             "categórica",        "Editora do livro (Google Books)", "-"),
    ("tem_avaliacoes_google_books", "booleana",  "Indica se o livro tem avaliação no Google Books", "-"),
    ("titulo_tmdb",         "texto",             "Título do filme conforme a TMDB", "-"),
    ("data_lancamento",     "data/hora",         "Data de lançamento do filme (TMDB)", "data"),
    ("ano_lancamento_filme","numérica discreta", "Ano de lançamento do filme (derivado de data_lancamento)", "ano"),
    ("idioma_original",     "categórica",        "Idioma original do filme (TMDB)", "-"),
    ("generos",             "categórica",        "Gêneros do filme (TMDB)", "-"),
    ("duracao_min",         "numérica contínua", "Duração do filme (0 tratado como não divulgado -> NaN)", "minutos"),
    ("orcamento",           "numérica contínua", "Orçamento do filme (0 tratado como não divulgado -> NaN)", "USD nominal"),
    ("receita",             "numérica contínua", "Receita/bilheteria do filme (0 -> NaN)", "USD nominal"),
    ("popularidade",        "numérica contínua", "Índice de popularidade da TMDB", "score TMDB"),
    ("media_votos",         "numérica contínua", "Nota média do filme na TMDB", "escala 0–10"),
    ("contagem_votos",      "numérica discreta", "Número de votos do filme na TMDB", "contagem"),
    ("tem_match_tmdb",      "booleana",          "Indica se o título encontrou um filme na TMDB", "-"),
    ("sim_titulo_gb",       "numérica contínua", "Similaridade (0–100) entre título do livro (Google Books) e da Wikipédia", "score 0–100"),
    ("sim_titulo_tmdb",     "numérica contínua", "Similaridade (0–100) entre título do filme (TMDB) e da Wikipédia", "score 0–100"),
    ("ano_coerente",        "booleana",          "True se o livro foi publicado até 3 anos após o filme (coerência de datas)", "-"),
    ("match_confiavel",     "booleana",          "Match validado: TMDB casou + título similar + ano coerente", "-"),
    ("retorno_financeiro",  "numérica contínua", "(receita - orçamento) / orçamento, quando orçamento > 0", "razão"),
]
df_dicionario = pd.DataFrame(dicionario, columns=["Variável", "Tipo", "Descrição", "Unidade"])
# Confere se cobre exatamente as colunas da base (evita esquecer/sobrar coluna)
faltando = set(df_integrado.columns) - set(df_dicionario["Variável"])
sobrando = set(df_dicionario["Variável"]) - set(df_integrado.columns)
if faltando:
    print("ATENÇÃO — colunas na base sem entrada no dicionário:", faltando)
if sobrando:
    print("ATENÇÃO — entradas no dicionário que não existem na base:", sobrando)
if not faltando and not sobrando:
    print("Dicionário cobre exatamente as", len(df_integrado.columns), "colunas da base.")
df_dicionario.to_csv(TRATADO / "dicionario_variaveis.csv", index=False, encoding="utf-8")
display(df_dicionario)

Dicionário cobre exatamente as 29 colunas da base.


,Variável,Tipo,Descrição,Unidade
0,chave_titulo,texto,Título normalizado (sem acento/caixa/pontuação...,-
1,titulo_wikipedia,texto,Título da obra como listado na categoria da Wi...,-
2,url_artigo,texto,URL do artigo na Wikipédia (atribuição da fonte),-
3,titulo_google_books,texto,Título do livro conforme a Google Books API,-
4,autor,categórica,Autor(es) do livro (Google Books),-
5,ano_publicacao_livro,numérica discreta,Ano de publicação da edição retornada pelo Goo...,ano
6,nota_media,numérica contínua,Nota média do livro no Google Books,escala 0–5
7,n_avaliacoes,numérica discreta,Número de avaliações do livro no Google Books,contagem
8,categorias,categórica,Categorias/gêneros do livro (Google Books),-
9,idioma,categórica,Idioma da edição do livro (Google Books),-


In [ ]:
# Salvando a base tratada (integrada e limpa) — separada do bruto, que segue intocado em dados_brutos/
caminho_csv_tratado = TRATADO / "base_livros_filmes_tratada.csv"
df_integrado.to_csv(caminho_csv_tratado, index=False, encoding="utf-8")

# Parquet é o formato preferencial pedido no enunciado (preserva tipos, mais compacto)
caminho_parquet_tratado = TRATADO / "base_livros_filmes_tratada.parquet"
df_integrado.to_parquet(caminho_parquet_tratado, index=False)

salvar_proveniencia()

print(f"Salvo: {caminho_csv_tratado}")
print(f"Salvo: {caminho_parquet_tratado}")

# Prova de que os dados brutos continuam intocados
print("\nBrutos preservados e intocados:")
print(f"  df_wikipedia_bruto:    {df_wikipedia_bruto.shape}")
print(f"  df_google_books_bruto: {df_google_books_bruto.shape}")
print(f"  df_tmdb_bruto:         {df_tmdb_bruto.shape}")


Proveniência salva em: dados_brutos/proveniencia.json (1216 entradas)
Salvo: dados_tratados/base_livros_filmes_tratada.csv
Salvo: dados_tratados/base_livros_filmes_tratada.parquet

Brutos preservados e intocados:
  df_wikipedia_bruto:    (424, 3)
  df_google_books_bruto: (424, 9)
  df_tmdb_bruto:         (424, 12)


### 10. Visualização completa da base tratada

Exibindo a base tratada por inteiro (todas as linhas e colunas), para conferência final antes de
seguir para as próximas fases da disciplina.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

df_integrado

,chave_titulo,titulo_wikipedia,url_artigo,titulo_google_books,autor,ano_publicacao_livro,nota_media,n_avaliacoes,categorias,idioma,editora,tem_avaliacoes_google_books,titulo_tmdb,data_lancamento,ano_lancamento_filme,idioma_original,generos,duracao_min,orcamento,receita,popularidade,media_votos,contagem_votos,tem_match_tmdb,sim_titulo_gb,sim_titulo_tmdb,ano_coerente,match_confiavel,retorno_financeiro
0,artur e os minimeus,Artur e os Minimeus,https://pt.wikipedia.org/wiki/Artur_e_os_Minim...,Arthur and the Minimoys,Luc Besson,2006.0,4.0,1.0,Juvenile Fiction,en,Harper Collins,True,Arthur e os Minimoys,2006-12-13,2006.0,fr,"Aventura, Fantasia, Animação, Família",94.0,86000000.0,1.086056e+08,6.5825,6.394,3100.0,True,61.904762,87.179487,True,True,2.628559e-01
1,1 litre no namida,1 Litre no Namida,https://pt.wikipedia.org/wiki/1_Litre_no_Namid...,1 Litre of Tears,Kito Aya,2024.0,NaN,NaN,Biography & Autobiography,vi,Não informado,False,1リットルの涙,2005-02-05,2005.0,ja,Drama,98.0,NaN,NaN,1.9354,7.833,36.0,True,54.545455,8.333333,False,False,NaN
2,84 charing cross road,84 Charing Cross Road,https://pt.wikipedia.org/wiki/84_Charing_Cross...,"84, Charing Cross Road",Helene Hanff,2027.0,NaN,NaN,Literary Collections,en,Penguin Classics,False,"Nunca Te Vi, Sempre Te Amei",1987-02-13,1987.0,en,"Drama, Romance",100.0,NaN,2.538291e+06,2.9330,7.200,232.0,True,97.674419,25.000000,False,False,NaN
3,2010 odyssey two,2010: Odyssey Two,https://pt.wikipedia.org/wiki/2010:_Odyssey_Two,2010: Odyssey Two,Arthur C. Clarke,1984.0,4.0,18.0,Fiction,en,Del Rey,True,2010: O Ano em que Faremos Contato,1984-12-06,1984.0,en,"Thriller, Ficção científica",115.0,28000000.0,4.040000e+07,12.2490,6.662,1117.0,True,100.000000,39.215686,True,False,4.428571e-01
4,3096 dias,3096 Dias,https://pt.wikipedia.org/wiki/3096_Dias,"3,096 Days","Natascha Kampusch, Heike Gronemeier, Corinna M...",2010.0,NaN,NaN,Biography & Autobiography,en,Penguin UK,False,3096 Dias de Cativeiro,2013-02-21,2013.0,de,Drama,111.0,NaN,6.677474e+06,8.3553,7.398,1022.0,True,84.210526,58.064516,True,False,NaN
5,a autobiografia de um mentiroso volume vi,A Autobiografia de um Mentiroso: Volume VI,https://pt.wikipedia.org/wiki/A_Autobiografia_...,Vi-a Mais Sacra,Melkyzedek Siqueira César,2024.0,NaN,NaN,Biography & Autobiography,pt-BR,Clube de Autores,False,None,NaT,NaN,Não informado,Não informado,NaN,NaN,NaN,NaN,NaN,NaN,False,28.070175,NaN,False,False,NaN
6,a culpa e das estrelas,A Culpa É das Estrelas,https://pt.wikipedia.org/wiki/A_Culpa_%C3%89_d...,A Culpa Não É Das Estrelas,Daniel Carlos,2022.0,NaN,NaN,Education,pt-BR,Clube de Autores,False,A Culpa é das Estrelas,2014-06-03,2014.0,en,"Romance, Drama",132.0,14000000.0,3.071668e+08,8.7213,7.590,11686.0,True,91.666667,100.000000,False,False,2.094049e+01
7,a espera de um milagre,À Espera de um Milagre,https://pt.wikipedia.org/wiki/%C3%80_Espera_de...,À Espera Da Extraordinária Cura,Marinalva Irenice Conceição; Guilherme Do Espí...,2025.0,NaN,NaN,Humor,pt-BR,Clube de Autores,False,À Espera de um Milagre,1999-12-10,1999.0,en,"Fantasia, Drama, Crime",189.0,60000000.0,2.868014e+08,35.0176,8.506,19795.0,True,49.056604,100.000000,False,False,3.780023e+00
8,a feiticeira,A Feiticeira,https://pt.wikipedia.org/wiki/A_Feiticeira_(li...,"Winnie, a Feiticeira","Korky Paul, Valerie Thomas",1998.0,NaN,NaN,Cats,pt-BR,Martins Martins Fontes,False,A Feiticeira,2005-06-21,2005.0,en,"Comédia, Fantasia, Romance",102.0,85000000.0,1.314262e+08,5.7212,5.091,1689.0,True,75.000000,100.000000,True,True,5.461902e-01
9,a filosofia na alcova,A Filosofia na Alcova,https://pt.wikipedia.org/wiki/A_Filosofia_na_A...,"Filosofia na Alcôva, A",marquis de Sade,2000.0,5.0,1.0,Fiction,pt-BR,Editora Iluminuras Ltda,True,A Filosofia na Alcova,2017-11-23,2017.0,pt,Drama,77.0,NaN,NaN,0.9440,5.625,8.0,True,93.023256,100.000000,True,True,NaN


In [ ]:
!zip -r dados_brutos.zip dados_brutos
!zip -r dados_tratados.zip dados_tratados

updating: dados_brutos/ (stored 0%)
updating: dados_brutos/tmdb_resultados.csv (deflated 59%)
updating: dados_brutos/tmdb_json/ (stored 0%)
updating: dados_brutos/tmdb_json/tmdb_detalhes_0266.json (deflated 45%)
updating: dados_brutos/tmdb_json/tmdb_detalhes_0287.json (deflated 44%)
updating: dados_brutos/tmdb_json/tmdb_busca_0390.json (deflated 30%)
updating: dados_brutos/tmdb_json/tmdb_detalhes_0092.json (deflated 47%)
updating: dados_brutos/tmdb_json/tmdb_busca_0359.json (deflated 57%)
updating: dados_brutos/tmdb_json/tmdb_busca_0024.json (deflated 52%)
updating: dados_brutos/tmdb_json/tmdb_detalhes_0225.json (deflated 42%)
updating: dados_brutos/tmdb_json/tmdb_busca_0347.json (deflated 34%)
updating: dados_brutos/tmdb_json/tmdb_busca_0183.json (deflated 55%)
updating: dados_brutos/tmdb_json/tmdb_busca_0418.json (deflated 26%)
updating: dados_brutos/tmdb_json/tmdb_busca_0104.json (deflated 66%)
updating: dados_brutos/tmdb_json/tmdb_detalhes_0135.json (deflated 41%)
updating: dados_b